# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. We will:
- Load and inspect dataset metadata and record sets
- Extract and explore tabular data
- Perform data filtering and transformations
- Visualize patterns in the proteomics data

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Print high-level metadata
md = dataset.metadata
print(f"Dataset Name: {md.name}")
print(f"Description: {md.description}\n")
print(f"Published: {md.datePublished}")
print(f"License: {md.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).

We will enumerate record sets and their field IDs, as all Croissant entities are referenced with their `@id`.

In [ ]:
# List all available record sets and their fields by @id
print("Record Sets in the Dataset:\n")

record_sets = dataset.metadata.recordSets
for rs in record_sets:
    print(f"- Record Set: {rs['@id']} (name: {rs.get('name', 'N/A')})")
    fields = rs.get('fields', [])
    print("  Fields and Columns (by @id):")
    for field in fields:
        print(f"    - {field['@id']} (name: {field.get('name', 'N/A')}, column: {field.get('column', {}).get('@id','N/A')})")
    print()
# Store the list of record set IDs for later
record_set_ids = [rs['@id'] for rs in record_sets]

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Let's extract the primary record set (typically the first one listed)
if len(record_set_ids) == 0:
    raise RuntimeError("No record sets found in the dataset. Please check the Croissant schema.")

# Choose the main record set (commonly protein-level data)
primary_record_set_id = record_set_ids[0]
print(f"Extracting records from record set: {primary_record_set_id}\n")

dataframes = {}
for rs_id in record_set_ids:
    # Extract all records (each row is a dict mapping field @id to value)
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} rows for record set @id: {rs_id}")
    else:
        dataframes[rs_id] = pd.DataFrame()

# Show column (field) IDs and first few rows for the primary DataFrame
print("\nColumns (fields by @id):")
print(dataframes[primary_record_set_id].columns.tolist())

dataframes[primary_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's identify a numeric field (e.g., sequence coverage or peptide counts) and a group/category field (e.g., protein accession or modification group).

In [ ]:
# The actual @ids of fields will depend on the Croissant schema.
# Use the previous cell's output for your dataset. For example:
#   coverage_field_id = 'cr:field_coverage_percent'
#   protein_group_field_id = 'cr:field_modification_type'
# Let's try auto-detecting a numeric field:

df = dataframes[primary_record_set_id]
# Try auto-selecting a numeric field
numeric_field = None
for col in df.columns:
    # Heuristic: look for numbers in the column
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break
if numeric_field is None:
    # fallback: try columns with 'coverage' or 'abundance' in their @id
    for col in df.columns:
        if 'coverage' in col.lower() or 'abundance' in col.lower():
            numeric_field = col
            break
if numeric_field is None:
    raise RuntimeError("No numeric field found in the main record set.")
print(f"Selected numeric field (@id): {numeric_field}")

# Try to detect a categorical/group field
group_field = None
for col in df.columns:
    if col != numeric_field and df[col].nunique() < max(25, len(df)//10):
        group_field = col
        break
if group_field is not None:
    print(f"Selected group/categorical field (@id): {group_field}")

# Set a threshold for numeric filtering (e.g., top 25% quantile)
threshold = df[numeric_field].quantile(0.75)
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.3f} (top 25% quantile): {len(filtered_df)} rows\n")

filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records (first 5 rows):")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# If a group field is available, perform group mean/summary
if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"mean_{numeric_field}")
    print(f"\nGrouped mean {numeric_field} by {group_field} (first 5 groups):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field], bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

if group_field is not None:
    # Show top 10 groups for a boxplot
    top_groups = df[group_field].value_counts().index[:10]
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=group_field, y=numeric_field, data=df[df[group_field].isin(top_groups)])
    plt.title(f"{numeric_field} by {group_field} (Top 10 Groups)")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

- The dataset schema and record sets were successfully loaded using the `mlcroissant` library, referencing all entities by their `@id`s.
- We explored the main record set and visualized the distribution of a numeric field (e.g., sequence coverage or abundance).
- Filtering and normalization operations were applied for downstream analysis.
- Group-level summaries and boxplots can highlight biological or technical differences across categorical factors.

For further domain-specific analyses, consult the dataset documentation and scientific context referenced in the FAIR^2 schema.